# Requirement 22: Direct result delivery

> Return sensitive results through the protected session channel only.

**Covered by:** Caller

**How it is fulfilled:** The sandbox encrypts the result with the session key before it leaves, and the orchestrator decrypts it. The result is never written to shared storage or sent through another service.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
sandbox, session_id = new_session()
session_key = install_session_key(sandbox, session_id)
sandbox.write_file("/tmp/encrypt_result.py", '''
import os, sys
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
key = open("/tmp/session_key.bin", "rb").read()
nonce = os.urandom(12)
sid = sys.argv[1].encode()
open("/tmp/result.enc", "wb").write(nonce + AESGCM(key).encrypt(nonce, b"total=2000", sid))
''')
print(sandbox.exec(f"python /tmp/encrypt_result.py {session_id}"))

ExecResult(exit_code=0, stdout='', stderr='')


In [3]:
blob = sandbox.read_file("/tmp/result.enc")
print("what travels back:", blob.hex()[:48], "...")
check("the returned bytes are not readable plaintext", b"total" not in blob)
plain = AESGCM(session_key).decrypt(blob[:12], blob[12:], session_id.encode())
print("orchestrator decrypts:", plain)
check("orchestrator recovers the result", plain == b"total=2000")

what travels back: d9bae5d611cd859ed2816b6692cbb8aaf148e2d9d1a286ae ...
PASS  the returned bytes are not readable plaintext
orchestrator decrypts: b'total=2000'
PASS  orchestrator recovers the result


In [4]:
try:
    AESGCM(session_key).decrypt(blob[:12], blob[12:], b"other-session")
    check("wrong session binding rejected", False)
except Exception:
    check("wrong session binding rejected", True)
cleanup(sandbox)

PASS  wrong session binding rejected


**Caveat:** The result channel is protected by the session key. It does not stop a Data Owner holder from reading the file inside the sandbox, so the role restriction (requirement 1) still matters.